Split do guia inserido pelo parametro path:str

In [6]:
import io
from typing import List
from pypdf import PdfReader, PdfWriter

def split_pdf(path: str) -> List[bytes]:
    paginas_em_bytes = []
    
    reader = PdfReader(path)
    total_paginas = len(reader.pages)
    
    print(f"[Split] Cortando PDF com {total_paginas} páginas...")
    
    for idx in range(total_paginas):
        writer = PdfWriter()
        # Adiciona apenas a página atual ao novo escritor
        writer.add_page(reader.pages[idx])
        
        # Salva o resultado temporariamente na memória (em bytes)
        buffer = io.BytesIO()
        writer.write(buffer)
        
        # Recupera os bytes e adiciona à lista
        paginas_em_bytes.append(buffer.getvalue())
        buffer.close()
        
    print(f"[Split] Concluído! {len(paginas_em_bytes)} páginas isoladas em memória.")
    return paginas_em_bytes

Vetorização das páginas do pdf

In [7]:
from google import genai
from dotenv import load_dotenv

load_dotenv()

client = genai.Client()

def embed_page_bytes(page_bytes: bytes, client) -> list[float]:
    from google.genai import types
    
    response = client.models.embed_content(
        model="gemini-embedding-2-preview",
        contents=genai.types.Part.from_bytes(data=page_bytes, mime_type="application/pdf"),
        config=genai.types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
    )
    return response.embeddings[0].values


Setup do Chroma (banco vetorial local persistente)

In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_or_create_collection(
    name="guias_saude",
    metadata={"hnsw:space": "cosine"}, 
    embedding_function=None,
)

print(f"[Chroma] Coleção pronta. Itens atuais: {collection.count()}")

[Chroma] Coleção pronta. Itens atuais: 0


In [ ]:
import os

def pipeline(pdf_path: str):
    paginas = split_pdf(pdf_path)
    source_name = os.path.basename(pdf_path)

    ids, embeddings, metadatas = [], [], []

    for i, pagina_bytes in enumerate(paginas):
        num_pagina = i + 1
        print(f"[Pipeline] Vetorizando página {num_pagina}/{len(paginas)}...")

        vetor = embed_page_bytes(pagina_bytes, client)

        ids.append(f"{source_name}::page_{num_pagina}")
        embeddings.append(vetor)
        metadatas.append({
            "source": source_name,
            "path": pdf_path,
            "page": num_pagina,
        })

    collection.upsert(ids=ids, embeddings=embeddings, metadatas=metadatas)
    print(f"[Pipeline] {len(ids)} vetores salvos no Chroma. Total na coleção: {collection.count()}")


paths = [
    "guias/Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf",
    "guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf",
]

for guia in paths:
    pipeline(guia)

[Split] Cortando PDF com 122 páginas...
[Split] Concluído! 122 páginas isoladas em memória.
[Pipeline] Vetorizando página 1/122...
[Pipeline] Vetorizando página 2/122...
[Pipeline] Vetorizando página 3/122...
[Pipeline] Vetorizando página 4/122...
[Pipeline] Vetorizando página 5/122...
[Pipeline] Vetorizando página 6/122...
[Pipeline] Vetorizando página 7/122...
[Pipeline] Vetorizando página 8/122...
[Pipeline] Vetorizando página 9/122...
[Pipeline] Vetorizando página 10/122...
[Pipeline] Vetorizando página 11/122...
[Pipeline] Vetorizando página 12/122...
[Pipeline] Vetorizando página 13/122...
[Pipeline] Vetorizando página 14/122...
[Pipeline] Vetorizando página 15/122...
[Pipeline] Vetorizando página 16/122...
[Pipeline] Vetorizando página 17/122...
[Pipeline] Vetorizando página 18/122...
[Pipeline] Vetorizando página 19/122...
[Pipeline] Vetorizando página 20/122...
[Pipeline] Vetorizando página 21/122...
[Pipeline] Vetorizando página 22/122...
[Pipeline] Vetorizando página 23/122.

### Exemplo de consulta

In [ ]:
from google.genai import types

def query(pergunta: str, source: str | None = None, k: int = 3):
    resp = client.models.embed_content(
        model="gemini-embedding-2-preview",
        contents=pergunta,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    )
    q_vec = resp.embeddings[0].values

    where = {"source": source} if source else None
    return collection.query(query_embeddings=[q_vec], n_results=k, where=where)

res = query("Quais são os sintomas de hipoglicemia?",
            source="Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf")
for id_, meta, dist in zip(res["ids"][0], res["metadatas"][0], res["distances"][0]):
    print(f"  dist={dist:.4f}  {meta['source']}  pág. {meta['page']}  ({id_})")

  dist=0.5800  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 98  (Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf::page_98)
  dist=0.5813  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 8  (Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf::page_8)
  dist=0.5943  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 62  (Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf::page_62)


In [15]:
res = query("Classificação de risco do pé diabético",
            source="Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf")
for id_, meta, dist in zip(res["ids"][0], res["metadatas"][0], res["distances"][0]):
    print(f"  dist={dist:.4f}  {meta['source']}  pág. {meta['page']}  ({id_})")

  dist=0.5162  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 93  (Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf::page_93)
  dist=0.5221  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 94  (Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf::page_94)
  dist=0.5416  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 92  (Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf::page_92)
